In [ ]:
def main(datasources, start_date, end_date):
    """端到端单频率 Transformer 极简示例。

    赛制约定: 平台只替换 datasources / start_date / end_date, 其中
    start_date~end_date 为【测试集区间】。训练区间写死(TRAIN_START/END),
    用样本外的测试区间做预测, 输出每日分数 ['date','instrument','score']。
    切勿用传入的 start_date/end_date 训练(数据泄漏, 会被审查)。

    端到端理念: 不做显式因子工程, 直接把原始 1 分钟量价/盘口序列当 token 序列
    喂给 Transformer, 在收盘决策点取整段序列的表征送入回归头。
    仅用 10 个原始字段, 只做"按字段标准化 + 成交量 log"这类规则允许的预处理。
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置 (写死, 不随平台入参变化) ----------
    TABLE = datasources["bar1m"]    # 只用 1 分钟 K 线
    TRAIN_START, TRAIN_END = "2022-01-01", "2023-12-31 23:59:59"
    SEQ_LEN = 64          # 每条样本回看多少个 bar
    EPOCHS, BATCH, LR, SEED = 5, 512, 1e-3, 42
    MAX_TRAIN_INSTRUMENTS = 200   # demo 限制训练标的数控制时长, 正式可放开

    PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    VOL_COLS   = ["volume", "amount", "bid_volume1", "ask_volume1"]  # 量纲大, 先 log1p
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)

    np.random.seed(SEED); torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info("运行设备", device=str(device), table=TABLE)

    # ---------- 模型: 单条 Transformer 编码 -> 池化 -> 回归头 ----------
    class StockTransformer(nn.Module):
        def __init__(self, n_feat, d_model=64, nhead=4, nlayers=2, dim_ff=128, seq_len=SEQ_LEN):
            super().__init__()
            self.proj = nn.Linear(n_feat, d_model)                  # 每个 bar -> token 向量
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))  # 可学习位置编码
            layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, 0.1,
                                               batch_first=True, activation="gelu")
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

        def forward(self, x):                                       # (B, L, N_FEAT) -> (B,)
            h = self.encoder(self.proj(x) + self.pos).mean(dim=1)
            return self.head(h).squeeze(-1)

    # ---------- 数据: 直接用原始字段切窗口, 只做 量log + 标准化 ----------
    def build_dataset(sd, ed, mode, instruments, stats=None):
        """切窗口并标准化。
        mode='train' 返回 (X, y, None, stats); 'infer' 返回 (X, None, idx_df, stats)。
        X 为 (N, SEQ_LEN, N_FEAT); stats 为 (mean, std), 训练集上算好, 推理复用。"""
        t0 = time.time()
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")  # 缓冲凑回看窗口
        sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {TABLE} ORDER BY instrument, date"
        df = dai.query(sql, filters={"date": [buf, ed], "instrument": instruments}).df()
        for c in VOL_COLS:
            df[c] = np.log1p(df[c].clip(lower=0))                   # 量纲大的字段先 log1p

        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
        wins, ys, keys = [], [], []
        for ins, sub in df.groupby("instrument", sort=False):
            if len(sub) <= SEQ_LEN:
                continue
            feats = sub[FEATURE_COLS].to_numpy(np.float32)
            day = sub["date"].dt.normalize().to_numpy()            # 1m bar 时间戳取自然日
            close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))  # 每日最后一根 bar
            close_px = sub["close"].to_numpy(np.float64)[close_pos]
            dates = day[close_pos]
            for k, p in enumerate(close_pos):
                d = pd.Timestamp(dates[k])
                if p + 1 < SEQ_LEN or d < sd_ts or d > ed_ts:
                    continue                                        # 历史不足 或 落在缓冲区
                label = None
                if k + 1 < len(close_pos) and close_px[k] > 0:
                    r = close_px[k + 1] / close_px[k] - 1.0         # 未来 1 日收益
                    if np.isfinite(r):
                        label = np.float32(r)
                if mode == "train" and label is None:
                    continue                                        # 训练集需要标签
                wins.append(feats[p - SEQ_LEN + 1: p + 1])
                ys.append(label if label is not None else np.float32(0.0))
                keys.append((d, ins))
        if not keys:
            raise RuntimeError(f"build_dataset 无样本 (mode={mode}, {sd}~{ed})")

        X = np.stack(wins).astype(np.float32)                       # (N, SEQ_LEN, N_FEAT)
        if stats is None:                                           # 训练集上算, 推理复用
            flat = X.reshape(-1, N_FEAT)
            stats = (flat.mean(0).astype(np.float32), flat.std(0).astype(np.float32) + 1e-6)
        m, s = stats
        X = ((X - m) / s).astype(np.float32)                        # 按字段标准化
        logger.info(f"{mode} 集构建完成", samples=len(keys),
                    elapsed=round(time.time() - t0, 2))
        if mode == "train":
            return X, np.array(ys, np.float32), None, stats
        return X, None, pd.DataFrame(keys, columns=["date", "instrument"]), stats

    def pool(sd, ed):
        """区间内中证 1000 成分股代码。"""
        df = dai.query("SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
                       filters={"date": [sd, ed]}).df()
        return df["instrument"].tolist()

    # ---------- 训练 (写死训练区间, 从零训练) ----------
    logger.info("构建训练集", start=TRAIN_START, end=TRAIN_END)
    Xtr, ytr, _, stats = build_dataset(
        TRAIN_START, TRAIN_END, "train", pool(TRAIN_START, TRAIN_END)[:MAX_TRAIN_INSTRUMENTS])
    lo, hi = np.percentile(ytr, [1, 99]); ytr = np.clip(ytr, lo, hi)  # winsorize 标签

    model = StockTransformer(N_FEAT).to(device)
    logger.info("可训练参数量", n_params=sum(p.numel() for p in model.parameters()))
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                        batch_size=BATCH, shuffle=True, pin_memory=(device.type == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    model.train()
    for ep in range(EPOCHS):
        t, tot, nb = time.time(), 0.0, 0
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        logger.info("epoch 完成", epoch=ep + 1, mse=round(tot / max(nb, 1), 8),
                    elapsed=round(time.time() - t, 2))

    # ---------- 推理 (样本外测试区间) ----------
    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    Xte, _, idx_df, _ = build_dataset(start_date, end_date, "infer", pool(start_date, end_date), stats)
    model.eval()
    preds = []
    Xte_t = torch.from_numpy(Xte)
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = Xte_t[i:i + BATCH].to(device)
            preds.append(model(xb).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    # 只用 1 分钟 K 线作为输入数据
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
    }

    # 本地用一小段区间模拟「平台注入的测试集区间」(训练区间已在 main 内写死)
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )